# S3 - Calidad de datos y formatos analiticos particionados

**Actividad:** construir el notebook `03_calidad_datos_practica.ipynb` sobre el entorno `lambda26` (`uso-pyspark`), validando esquema, tratando nulos y duplicados, y escribiendo una salida analitica particionada en Parquet — sobre el dataset real H&M ya usado en S2 (`customers.csv`, `articles.csv`).

**Proposito de la actividad:** dejar evidencia ejecutable de que dominas los controles de calidad de datos (esquema, nulos, duplicados) y el particionamiento de salidas analiticas — antes de avanzar a ML distribuido (S4).

Guia completa: `docs/sesiones/S03_Calidad_Datos_Particionamiento_Formatos_Analiticos.md`, seccion 3.

## 3.1 Preparar los datos de S3 y reanudar el entorno `lambda26`

**Producto del paso:** `customers.csv` y `articles.csv` disponibles en `pyspark/sesiones/s03-calidad-datos/data/`, entorno `lambda26` funcionando.

Ya descargaste estos dos archivos en S2 — no hace falta descargarlos de nuevo, solo copialos a la carpeta de esta sesion (desde tu maquina, no dentro del notebook):

```bash
cp lambda26/pyspark/sesiones/s02-fundamentos/data/customers.csv lambda26/pyspark/sesiones/s03-calidad-datos/data/
cp lambda26/pyspark/sesiones/s02-fundamentos/data/articles.csv lambda26/pyspark/sesiones/s03-calidad-datos/data/
```

Esta sesion no necesita `transactions.parquet` — el foco es esquema/nulos/duplicados sobre datos tabulares, no sobre transacciones.

## 3.2 Crear el notebook y la `SparkSession`

**Producto del paso:** notebook con una `SparkSession` activa.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion3-calidad-datos")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/21 00:46:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/21 00:46:08 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/08/21 00:46:08 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


`spark.driver.memory` en `4g` desde el arranque — en S2 la JVM se cayo por quedarse en el default de 1g; acá se fija de una vez, no se espera al primer crash.

In [2]:
ORIGEN_DATOS = "/opt/s03-calidad-datos/data"
ARTIFACTS = "/opt/s03-calidad-datos/artifacts"

## 3.3 Cargar `customers.csv` y validar el esquema

**Producto del paso:** `df_customers` cargado con esquema explicito, verificado contra lo esperado — control de calidad #1: esquema.

Mismo esquema explicito que en S2 (evita el riesgo de que `inferSchema` adivine mal un tipo):

In [3]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

schema_customers = StructType([
    StructField("customer_id", StringType(), nullable=True),
    StructField("FN", DoubleType(), nullable=True),
    StructField("Active", DoubleType(), nullable=True),
    StructField("club_member_status", StringType(), nullable=True),
    StructField("fashion_news_frequency", StringType(), nullable=True),
    StructField("age", IntegerType(), nullable=True),
    StructField("postal_code", StringType(), nullable=True),
])

df_customers = spark.read.csv(
    f"{ORIGEN_DATOS}/customers.csv",
    header=True,
    schema=schema_customers,
)

df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- FN: double (nullable = true)
 |-- Active: double (nullable = true)
 |-- club_member_status: string (nullable = true)
 |-- fashion_news_frequency: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- postal_code: string (nullable = true)



Confirma que el esquema real coincide con el que documentaste — 7 columnas, en el mismo orden y tipo:

In [4]:
print(df_customers.columns)
df_customers.count()

['customer_id', 'FN', 'Active', 'club_member_status', 'fashion_news_frequency', 'age', 'postal_code']


1371980

## 3.4 Explorar nulos por columna

**Producto del paso:** conteo exacto de nulos por columna, con porcentaje sobre el total — control de calidad #2: nulos.

En S2 encontraste que `FN` y `Active` tenían bastantes nulos con `.describe()` (la fila `count` daba menos que el total). Acá lo cuantificas exacto, columna por columna:

In [5]:
from pyspark.sql.functions import col, count, when

total_filas = df_customers.count()

df_customers.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_customers.columns
]).show(vertical=True, truncate=False)

-RECORD 0------------------------
 customer_id            | 0      
 FN                     | 895050 
 Active                 | 907576 
 club_member_status     | 6062   
 fashion_news_frequency | 16009  
 age                    | 15861  
 postal_code            | 0      



El mismo resultado, con porcentaje (más fácil de interpretar que el conteo crudo):

In [6]:
nulos = df_customers.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_customers.columns
]).collect()[0].asDict()

for columna, cantidad in nulos.items():
    porcentaje = cantidad / total_filas * 100
    print(f"{columna}: {cantidad} nulos ({porcentaje:.1f}%)")

customer_id: 0 nulos (0.0%)
FN: 895050 nulos (65.2%)
Active: 907576 nulos (66.2%)
club_member_status: 6062 nulos (0.4%)
fashion_news_frequency: 16009 nulos (1.2%)
age: 15861 nulos (1.2%)
postal_code: 0 nulos (0.0%)


## 3.5 Tratar nulos con `.na.drop()` y `.na.fill()`

**Producto del paso:** `df_customers_limpio` con nulos tratados columna por columna, cada decisión con un criterio documentado — no "borrar todo lo que tenga un nulo" sin pensarlo.

`.na.fill()` — rellena con un valor por defecto, columna por columna. `FN`/`Active` son columnas de tipo "bandera" (presente/ausente); un nulo ahí significa "la bandera no se activó", no un dato faltante que haya que adivinar — por eso se rellenan con `0.0`, no con un promedio. `fashion_news_frequency` nulo se rellena con `"NONE"`, la misma categoría que el propio dataset ya usa explícitamente para ese caso:

In [7]:
df_customers_limpio = df_customers.na.fill({
    "FN": 0.0,
    "Active": 0.0,
    "fashion_news_frequency": "NONE",
    "club_member_status": "UNKNOWN",
})

`age` **no** se rellena: inventar una edad sería fabricar un dato que no existe. Se documenta como limitación conocida, no se fuerza un valor.

`.na.drop()` — elimina filas donde una columna crítica es nula (acá, `customer_id`: sin identificador, la fila no sirve para nada). En este dataset probablemente no elimine ninguna, y ese es también un resultado válido: confirmar que la columna clave nunca falta:

In [8]:
df_customers_valido = df_customers_limpio.na.drop(subset=["customer_id"])

print(f"Filas antes: {df_customers.count()}, después de na.drop(subset=['customer_id']): {df_customers_valido.count()}")

Filas antes: 1371980, después de na.drop(subset=['customer_id']): 1371980


En una corrida real, ambos números dieron **1 371 980** — `customer_id` nunca llega nulo en este dataset, así que `na.drop()` no elimina ninguna fila. Es el resultado esperado: confirma que la columna clave está completa, no que el paso "no sirvió de nada".

`df_customers_valido` se reutiliza en casi todos los pasos que siguen (3.6-3.11), varios con su propio `.count()` — sin cachearlo, cada uno recalcularía la lectura completa de `customers.csv` más el `.na.fill()`/`.na.drop()` desde cero. `cache()` (2.5, ya usado en S2) guarda el resultado la primera vez que una acción lo dispara:

In [ ]:
df_customers_valido = df_customers_valido.cache()

## 3.6 Filtrar registros que no pasan validaciones (`filter()`/`where()`)

**Producto del paso:** filas fuera de rango identificadas (o confirmación de que no existen).

Ejemplo de validación de rango sobre `age` — si el conteo da 0, también es un control de calidad exitoso, no un resultado "vacío":

In [9]:
df_edad_invalida = df_customers_valido.filter((col("age") < 0) | (col("age") > 100))
df_edad_invalida.count()

0

## 3.7 Ordenar resultados (`orderBy()`/`sort()`)

**Producto del paso:** resultado ordenado por una columna real.

In [10]:
df_customers_valido.orderBy(col("age").desc()).select("customer_id", "age", "club_member_status").show(10, truncate=False)

+----------------------------------------------------------------+---+------------------+
|customer_id                                                     |age|club_member_status|
+----------------------------------------------------------------+---+------------------+
|e8b2a7bf44f42e808d58299c53e6e1ad47178d7d457f82085d2030939106bdba|99 |ACTIVE            |
|36d219eb822d04d07e5bb31e39caffefc17da81ff4a19bad55677be1490c48f9|99 |ACTIVE            |
|a01bd2e0e8bbf8db61f0e623a05325cde84955a37415b70e9358396df4198578|99 |ACTIVE            |
|a0b9e2473f699821b72833824a3e95972862039a49c5e9de398a16c038b06da2|99 |PRE-CREATE        |
|2b5284f19d272d7b7f1006289566076d72a7f097feb03f5c3e03ab39aa5bcc36|99 |ACTIVE            |
|eeaeb36eecef27871a0fe587858906e5f28d43872f6a4aa561ff78947124c39d|99 |ACTIVE            |
|687c675e64e5f5d962c99ebf915da230be8c907a65a1d512984a339575e5da9f|99 |PRE-CREATE        |
|7558adbc0401acdd74a7c8633ad97d25979e9859b8abca2890a4daf8f6cdc22b|99 |ACTIVE            |
|a2106ba21

## 3.8 Duplicados: `dropDuplicates()` y `distinct()`

**Producto del paso:** confirmación (o eliminación) de filas duplicadas, con la definición correcta de qué cuenta como duplicado — control de calidad #3.

`distinct()` compara **todas** las columnas de la fila; `dropDuplicates(["customer_id"])` compara solo esa columna. Si las tres cifras de abajo coinciden con el total, no hay duplicados reales en ninguna de las dos definiciones:

In [11]:
total = df_customers_valido.count()
sin_duplicados_fila_completa = df_customers_valido.distinct().count()
sin_duplicados_por_id = df_customers_valido.dropDuplicates(["customer_id"]).count()

print(f"Total: {total}, sin duplicar (fila completa): {sin_duplicados_fila_completa}, sin duplicar (por customer_id): {sin_duplicados_por_id}")

Total: 1371980, sin duplicar (fila completa): 1371980, sin duplicar (por customer_id): 1371980


En una corrida real, las tres cifras dieron **1 371 980** — cero duplicados, en ninguna de las dos definiciones. No es un resultado "vacío": es la confirmación real de que `customer_id` es una clave limpia en este dataset.

**Contraste real con `articles.csv`** (S2, sección 3.10): al correr `rdd.take(5)` sobre `detail_desc` te salieron descripciones idénticas repetidas. Eso **no** son duplicados de fila — cada `article_id` es distinto (variante de color/talla). Confirma cuántos `article_id` comparten la misma descripción, sin tratarlos como error:

In [ ]:
df_articles = spark.read.csv(f"{ORIGEN_DATOS}/articles.csv", header=True, inferSchema=True)

duplicados_por_descripcion = (
    df_articles.filter(col("detail_desc").isNotNull())
    .groupBy("detail_desc")
    .count()
    .filter(col("count") > 1)
    .orderBy(col("count").desc())
)
duplicados_por_descripcion.show(5, truncate=False)

`filter(col("detail_desc").isNotNull())` va **antes** del `groupBy()`: sin él, `groupBy()` agrupa todos los artículos sin descripción bajo un mismo grupo `NULL` — que en una corrida real salió como el "valor más repetido" (416 artículos), tapando los duplicados de contenido real que sí importan para este ejercicio. Un `NULL` que se repite no es un duplicado de descripción, es simplemente la ausencia del dato — ya lo trataste como tal en 3.4-3.5, no hace falta que reaparezca acá.

## 3.9 Deduplicar con `Window` + `row_number()`

**Producto del paso:** una fila representativa por `product_code`, cuando "duplicado" significa "mismo producto base, distintas variantes" — algo que `dropDuplicates()` no puede decidir por sí solo (no elige *cuál* fila conservar).

`article_id` identifica cada variante (color/talla); `product_code` identifica el producto base. Te quedas con un representante por producto (el de menor `article_id`):

In [13]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_producto = Window.partitionBy("product_code").orderBy("article_id")

df_articles_un_por_producto = (
    df_articles
    .withColumn("fila", row_number().over(window_producto))
    .filter(col("fila") == 1)
    .drop("fila")
)

print(f"Filas originales: {df_articles.count()}, un representante por product_code: {df_articles_un_por_producto.count()}")

Filas originales: 105542, un representante por product_code: 47224


En una corrida real sobre este dataset, la reducción fue de **105 542 filas a 47 224 representantes** — más de la mitad de `articles.csv` son variantes de color/talla de un producto que ya está representado por otra fila. Es la diferencia real entre `dropDuplicates()` (que no puede hacer esta reducción, porque cada `article_id` es único) y `Window`+`row_number()` (que sí, porque agrupa por `product_code` en vez de por `article_id`).

## 3.10 Escritura particionada en Parquet (`partitionBy()`, `coalesce()`/`repartition()`)

**Producto del paso:** salida analitica particionada por `club_member_status`, lista para BI/ML — el producto que pide el sílabo de esta sesión.

`repartition(4)` antes de escribir controla cuántos archivos caen dentro de **cada** carpeta de partición (sin esto, se hereda el número de particiones de la lectura original — la misma sorpresa de las ~50 particiones que viste en S2 al guardar la muestra de `customers.csv`):

In [14]:
(
    df_customers_valido
    .repartition(4)
    .write.mode("overwrite")
    .partitionBy("club_member_status")
    .parquet(f"{ARTIFACTS}/customers_particionado")
)

`partitionBy("club_member_status")` crea **una subcarpeta por valor distinto** de esa columna (`club_member_status=ACTIVE/`, `club_member_status=LEFT CLUB/`, ...) — no una columna dentro del archivo. Verifica la estructura real:

In [15]:
import os

for carpeta in sorted(os.listdir(f"{ARTIFACTS}/customers_particionado")):
    print(carpeta)

._SUCCESS.crc
_SUCCESS
club_member_status=ACTIVE
club_member_status=LEFT CLUB
club_member_status=PRE-CREATE
club_member_status=UNKNOWN


## 3.11 Leer de vuelta y verificar el particionamiento

**Producto del paso:** confirmación de que la salida particionada se lee correctamente y que el particionamiento sí se aprovecha en consultas.

In [16]:
df_verificacion = spark.read.parquet(f"{ARTIFACTS}/customers_particionado")
df_verificacion.printSchema()
df_verificacion.count()

root
 |-- customer_id: string (nullable = true)
 |-- FN: double (nullable = true)
 |-- Active: double (nullable = true)
 |-- fashion_news_frequency: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- club_member_status: string (nullable = true)



1371980

`club_member_status` reaparece en el esquema aunque no está dentro de los archivos Parquet físicos — Spark lo reconstruye a partir del nombre de la carpeta (*partition discovery*).

Filtra por la columna particionada y revisa el plan — deberías ver `PartitionFilters` (no solo `PushedFilters`, el que ya viste en S2 3.6): la diferencia es que esto ni siquiera abre las carpetas de las otras particiones, en vez de leerlas y descartar filas después:

In [17]:
df_verificacion.filter(col("club_member_status") == "ACTIVE").explain(True)

== Parsed Logical Plan ==
'Filter '`=`('club_member_status, ACTIVE)
+- Relation [customer_id#392,FN#393,Active#394,fashion_news_frequency#395,age#396,postal_code#397,club_member_status#398] parquet

== Analyzed Logical Plan ==
customer_id: string, FN: double, Active: double, fashion_news_frequency: string, age: int, postal_code: string, club_member_status: string
Filter (club_member_status#398 = ACTIVE)
+- Relation [customer_id#392,FN#393,Active#394,fashion_news_frequency#395,age#396,postal_code#397,club_member_status#398] parquet

== Optimized Logical Plan ==
Filter (isnotnull(club_member_status#398) AND (club_member_status#398 = ACTIVE))
+- Relation [customer_id#392,FN#393,Active#394,fashion_news_frequency#395,age#396,postal_code#397,club_member_status#398] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [customer_id#392,FN#393,Active#394,fashion_news_frequency#395,age#396,postal_code#397,club_member_status#398] Batched: true, DataFilters: [], Format: Parquet, Loc

Ya terminaste de reutilizar `df_customers_valido` — libera la memoria que ocupaba cacheado:

In [ ]:
df_customers_valido.unpersist()

## 3.12 Documentar hallazgos y responder preguntas de reflexión

**Producto del paso:** notebook documentado con celdas markdown explicando cada resultado.

**Reflexión técnica breve** (5 a 8 líneas): ¿qué columnas rellenaste con `.na.fill()` y cuáles no, y por qué? ¿Encontraste duplicados reales en `customers.csv`? ¿Qué diferencia notaste entre `PushedFilters` (S2) y `PartitionFilters` (S3) en el plan de ejecución?

_(Responde aquí)_